# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the FAIR^2 dataset package via its Croissant schema using the `mlcroissant` library.

### Dataset Source
- Croissant schema URL: https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

This notebook follows best practices by referencing all dataset entities (record sets, fields, columns, etc.) by their `@id`.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading

We load the dataset and inspect its metadata using `mlcroissant`. The dataset is described by a Croissant JSON-LD manifest, which specifies its structure and fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset loaded!')
print('\n--- Metadata Overview ---\n')
print('Name:', metadata.name)
print('Identifier:', getattr(metadata, 'identifier', '(no identifier)'))
print('Version:', getattr(metadata, 'version', '(no version)'))
print('Description:', metadata.description)
print('License:', getattr(metadata, 'license', '(no license)'))
print('Authors:', getattr(metadata, 'author', '(no authors specified)'))
print('Date Published:', getattr(metadata, 'datePublished', '(unknown)'))

## 2. Data Overview

List all available record sets and their fields, referencing each by its `@id`. This helps in identifying what data is available for extraction and exploration.

In [ ]:
# List all record sets' @id, names, and their field @id's and names
print('--- RecordSet Overview ---')
record_sets = list(dataset.record_sets)
if len(record_sets) == 0:
    print("No record sets found in the dataset metadata.")
else:
    for rs in record_sets:
        print(f"\nRecordSet: @id={rs.id}")
        print(f"  name: {getattr(rs, 'name', '(no name)')}")
        print('  Fields:')
        for fld in rs.fields:
            print(f"    - @id={fld.id}\tname={getattr(fld, 'name', '(no name)')}")

## 3. Data Extraction

We'll extract data from all available record sets by referencing each by its `@id`. For this dataset, we will attempt to load each record set into a pandas DataFrame so we can explore their schema and preview their contents.

In [ ]:
# Extract data from each record set by its @id
record_sets = list(dataset.record_sets)  # List[mlc.RecordSet]
dataframes = {}

if len(record_sets) == 0:
    print("No record sets available in the dataset.")
else:
    for rs in record_sets:
        # Use the record set's @id for reference
        rs_id = rs.id
        try:
            print(f"\nExtracting records for RecordSet @id: {rs_id}")
            records = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f'Loaded {len(df)} rows, columns: {df.columns.tolist()}')
            # Preview the first 3 rows
            display(df.head(3))
        except Exception as e:
            print(f"Failed to load record set {rs_id}: {e}")
    # For downstream: pick the first available record set if any
    if len(dataframes) > 0:
        main_recordset_id = list(dataframes.keys())[0]
        print(f"\nMain analysis will proceed with RecordSet: {main_recordset_id}")
    else:
        main_recordset_id = None

## 4. Exploratory Data Analysis (EDA)

Let's perform some basic data analysis:
- Filter records by a chosen numeric field (e.g., age at diagnosis or interval between diagnoses), referencing fields by their `@id`.
- Normalize the selected numeric field.
- Group data by a categorical field (e.g., sex, cancer type, or MSI-H status), again via `@id`.

This prepares the data for further investigation.

In [ ]:
# Choose field @id's by inspecting above output for available fields of interest
# (these should be replaced by the actual @id values from your dataset; here we guess likely field names for illustration)

import numpy as np

if 'main_recordset_id' not in locals() or main_recordset_id is None or len(dataframes) == 0:
    print("No main record set available for EDA.")
else:
    df = dataframes[main_recordset_id]

    # Attempt to find a numeric field (e.g., age or interval field).
    numeric_candidates = [col for col in df.columns if (
        ('age' in col.lower() or 'interval' in col.lower() or 'years' in col.lower()) 
        and df[col].dtype in [int, float, np.int64, np.float64]
    )]
    # Fallback: any numeric column
    if not numeric_candidates:
        numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Using numeric field for analysis: {numeric_field_id}")

        # Threshold arbitrarily set (adjust to suit the actual variable range)
        threshold = df[numeric_field_id].quantile(0.7) if not df[numeric_field_id].isnull().all() else 0

        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (top 5 shown):")
        display(filtered_df[[numeric_field_id]].head())

        # Normalize (z-score)
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
        print(f"Normalized {numeric_field_id} for filtered records (top 5 shown):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a group field: look for possible categorical variables ('sex', 'msi', 'site', 'type', etc.)
        group_candidates = [col for col in df.columns if (
            ('sex' in col.lower() or 'msi' in col.lower() or 'site' in col.lower() or 'type' in col.lower() or 'status' in col.lower()) 
            and (pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]))
        )]
        if group_candidates:
            group_field_id = group_candidates[0]
            print(f"Grouping field for analysis: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['count','mean','min','max'])
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field found for EDA in this record set.")

## 5. Visualization

Explore the distribution of a numeric field, and optionally, break down by a categorical grouping variable, referencing columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'main_recordset_id' not in locals() or main_recordset_id is None or len(dataframes) == 0:
    print("No main record set available for visualization.")
else:
    df = dataframes[main_recordset_id]
    # Use the same numeric and group field IDs from above if defined
    # (If not, attempt to select again)
    if 'numeric_field_id' not in locals():
        numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
        numeric_field_id = numeric_candidates[0] if numeric_candidates else None
    if numeric_field_id:
        plt.figure(figsize=(8,5))
        sns.histplot(df[numeric_field_id].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.show()

        # If a grouping field is available, plot boxplot
        if 'group_field_id' in locals() and group_field_id in df.columns:
            plt.figure(figsize=(10,6))
            sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.xlabel(group_field_id)
            plt.ylabel(numeric_field_id)
            plt.show()

## 6. Conclusion

This notebook demonstrated how to explore the FAIR^2 dataset using the `mlcroissant` library, referencing all dataset entities by their `@id` as recommended. We:

- Loaded the dataset via the Croissant schema and browsed available record sets
- Inspected dataset metadata and structure
- Loaded and previewed records from the main record set
- Performed basic exploratory data analysis (filtering, normalization, grouping)
- Visualized data distributions

For further analysis, refer to the `mlcroissant` documentation and use exact `@id` values exposed in the Croissant metadata for your advanced workflow.